# 02 — A two-line latency model has a model-dependent breakpoint.

The additive selection objective assigns a cost $T[|C|]$ to each contiguous chunk. Replacing that table by a structured function can simplify the optimization, but the boundary in the function need not coincide with a physical saturation point.

For the saved profile, **240 KiB** minimizes measured cost per byte. A two-line model fixed there has **3.80%** holdout MAPE; fitting its boundary instead selects **211 KiB** and gives **3.33%**. These results concern the approximation of the table, not a change in the measured read times.

This notebook contains the complete analysis. Set the CSV path below; no project Python modules or hardware measurements are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import nnls

CSV_PATH = "runs/02_chunk_latency/block_estimates.csv"
data = pd.read_csv(CSV_PATH)

## The saved measurements define an amortized cost.

Let $x>0$ be logical chunk length in KiB. In measurement block $b$, the chunk-count sweep gives a throughput estimate $\widehat B_b(x)$ and the corresponding cost

$$\widehat T_b(x)=\frac{1000x}{1024\widehat B_b(x)}\quad\text{ms/chunk}.$$

This is a throughput-derived amortized cost, not the service time of one isolated read. The CSV already contains these estimates. Define $T(x)$ as the median over all seven blocks. The first four blocks define $T_{\rm train}$; the last three define $T_{\rm test}$. Coefficients and free breakpoints use the training curve only.

In [ ]:
wide = data.pivot(index="size_kib", columns="block", values="latency_ms")
wide = wide.sort_index().sort_index(axis=1)
x, samples = wide.index.to_numpy(), wide.to_numpy()
T = np.median(samples, axis=1)
T_train = np.median(samples[:, :4], axis=1)
T_test = np.median(samples[:, 4:], axis=1)

## The minimum cost per byte is not a sustained throughput plateau.

Define the cost per KiB and its minimizing length by

$$\rho(x)=\frac{T(x)}x,\qquad x_\rho\in\arg\min_{x\in\mathcal G}\rho(x).$$

Since $B(x)=1000/(1024\rho(x))$, minimizing cost per byte is equivalent to maximizing throughput on the same curve. The alternative first-crossing rule uses a three-point median $\widetilde B_{\rm train}$:

$$s_{99}=\min\{x\in\mathcal G:\widetilde B_{\rm train}(x)\ge0.99\max_{u\in\mathcal G}\widetilde B_{\rm train}(u)\}.$$

These are different operations on different summaries of the same run.

In [ ]:
rho = T / x
x_rho = x[np.argmin(rho)]
B = 1000 * x / (1024 * T_train)
B_smooth = pd.Series(B).rolling(3, center=True, min_periods=1).median().to_numpy()
s99 = x[np.flatnonzero(B_smooth >= .99 * B_smooth.max())[0]]
pd.Series({"Minimum cost per KiB": x_rho, "First 99% crossing": s99}, name="KiB")

In [ ]:
zoom = x >= 160
plt.figure(figsize=(9, 3.5))
plt.plot(x[zoom], 1000 * rho[zoom], label="Seven-block median")
plt.plot(x[zoom], 1000 * T_test[zoom] / x[zoom], ":", label="Holdout")
plt.axvline(x_rho, ls="--", label=f"Raw minimum: {x_rho:g} KiB")
plt.axvline(s99, ls=":", label=f"99% crossing: {s99:g} KiB")
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Amortized cost / length (µs/KiB)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 1.** The measured minimum is **0.1660 µs/KiB at 240 KiB**. The view starts at 160 KiB to expose variation around the minimum rather than compressing it beneath the small-read costs.

The smoothed training curve first crosses 99% at **256 KiB**, but **125 of 129** measured sizes at or beyond that crossing fall below the same threshold. Over this interval, holdout $T/x$ has **4.03%** coefficient of variation and rises **8.36%** between its endpoints. A first crossing is not a durable plateau.

In [ ]:
tail = x >= s99
q = T_test[tail] / x[tail]
pd.Series({
    "Tail points": tail.sum(),
    "Below 99%": np.sum(B_smooth[tail] < .99 * B_smooth.max()),
    "Cost/length CV (%)": 100 * q.std(ddof=1) / q.mean(),
    "Endpoint change (%)": 100 * (q[-1] / q[0] - 1),
}).round(3)

## A boundary fixed at 240 KiB gives a useful two-line approximation.

Consider the continuous family, for $x>0$,

$$\boxed{T_s(x)=c_1x+\delta\max(s,x)
=\begin{cases}\delta s+c_1x,&x\le s,\\(c_1+\delta)x,&x\ge s,\end{cases}
\qquad c_1,\delta\ge0.}$$

Continuity ties the short-chunk intercept to the boundary, $a=\delta s$. The long-chunk line passes through the origin, so $T_s(x)/x=c_1+\delta$ is constant after $s$. This is a restriction on the model, not an observed identity.

For a fixed $s$, the design matrix has rows $(x,\max(s,x))$. Nonnegative least squares fits its two coefficients to $T_{\rm train}$. **Every model in this report has fitted coefficients; fixed and free refer only to the breakpoint.**

In [ ]:
def X_s(s):
    return np.column_stack((x, np.maximum(s, x)))


def T_s(s):
    return X_s(s) @ nnls(X_s(s), T_train)[0]


c1, delta = nnls(X_s(240), T_train)[0]
pd.Series({"Short intercept (ms)": 240 * delta,
           "Short slope (ms/KiB)": c1, "Long slope (ms/KiB)": c1 + delta})

For the 240 KiB boundary, the fitted function is approximately

$$T_{240}(x)=\begin{cases}0.011369+0.000136107x,&x\le240,\\0.000183478x,&x\ge240,\end{cases}\quad\text{ms}.$$

The calculations retain full precision. This fit anchors the model at the empirical efficiency minimum without claiming that the measured tail is exactly flat.

## Fitting the boundary is not the same as finding the efficiency minimum.

Let $\mathcal G^\circ$ be the measured grid without its first and last three sizes. The free boundary minimizes **training** error within the constrained family:

$$\boxed{s_\star\in\arg\min_{s\in\mathcal G^\circ}
\min_{c_1,\delta\ge0}\frac1{|\mathcal G|}\sum_{x\in\mathcal G}
[T_{\rm train}(x)-c_1x-\delta\max(s,x)]^2.}$$

The objective weights measured sizes equally. RMSE has the same minimizer as MSE and is easier to display in units of time.

In [ ]:
def rmse(observed, predicted):
    return np.sqrt(np.mean((observed - predicted) ** 2))


G = x[3:-3]
E_s = np.array([rmse(T_train, T_s(s)) for s in G])
s_star = G[np.argmin(E_s)]

A continuous hinge instead has the form

$$T_H(x)=a+bx+d(x-h)_+.$$

Its coefficients are unconstrained least-squares estimates, and $h$ is selected on the same training grid. For $x\ge h$, its tail is $(a-dh)+(b+d)x$, whose intercept need not vanish. A free-boundary constrained model has three parameters; a free hinge has four. Their boundaries therefore need not agree.

In [ ]:
def X_h(h):
    return np.column_stack((np.ones_like(x), x, np.maximum(x - h, 0)))


def T_h(h):
    return X_h(h) @ np.linalg.lstsq(X_h(h), T_train, rcond=None)[0]


E_h = np.array([rmse(T_train, T_h(h)) for h in G])
h_star = G[np.argmin(E_h)]
a, b, d = np.linalg.lstsq(X_h(h_star), T_train, rcond=None)[0]
band = G[E_s <= 1.01 * E_s.min()]
pd.Series({"Two-line boundary (KiB)": s_star, "Hinge boundary (KiB)": h_star,
           "Hinge tail intercept (µs)": 1000 * (a - d * h_star),
           "1% RMSE band, lower (KiB)": band.min(), "Upper (KiB)": band.max()}).round(3)

In [ ]:
local = (G >= 160) & (G <= 300)
plt.figure(figsize=(9, 3.5))
plt.plot(G[local], 1000 * E_s[local], label=f"Two-line; minimum at {s_star:g} KiB")
plt.plot(G[local], 1000 * E_h[local], "--", label=f"Hinge; minimum at {h_star:g} KiB")
for s in (s_star, 230, 240, 256):
    error = 1000 * rmse(T_train, T_s(s))
    plt.plot(s, error, "o", ms=4)
    plt.annotate(f"{s:g}", (s, error), xytext=(0, 8), textcoords="offset points", ha="center")
plt.xlabel("Candidate breakpoint (KiB)")
plt.ylabel("Training RMSE (µs/chunk)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 2.** Each point refits its family's coefficients at the candidate boundary. The circles mark the four nominated lengths in the constrained family; the other curve fits the free hinge. Neither search uses holdout error.

The constrained family selects **211 KiB**, while the hinge selects **230 KiB**. The hinge's fitted tail intercept is **−2.645 µs**; the constrained family's is zero. Fixing $T_s$ at 230 KiB does not turn it into the hinge model.

The constrained minimum is shallow: measured candidates within **1% of its minimum training RMSE** span **198–224 KiB**. This is an objective-sensitivity set, **not a confidence interval**. The value 211 KiB is a fitted approximation parameter, not a newly established optimal physical read size.

## Holdout errors compare the fitted functions, not SSD speeds.

Fit the coefficients again at each fixed boundary, and keep every fit frozen for evaluation. Include the affine baseline $T_A(x)=a+bx$. Define

$$\mathrm{RMSE}=\sqrt{\frac1{|\mathcal G|}\sum_x[\widehat T(x)-T_{\rm test}(x)]^2},\qquad
\mathrm{MAPE}=\frac{100}{|\mathcal G|}\sum_x\frac{|\widehat T(x)-T_{\rm test}(x)|}{T_{\rm test}(x)}.$$

In [ ]:
predictions = {}
for s in (240, 256, 230):
    predictions[f"Two-line ({s} KiB)"] = T_s(s)
free_line = f"Two-line ({s_star:g} KiB, free)"
free_hinge = f"Hinge ({h_star:g} KiB, free)"
predictions[free_line] = T_s(s_star)
predictions[free_hinge] = T_h(h_star)
X_a = np.column_stack((np.ones_like(x), x))
predictions["Affine"] = X_a @ np.linalg.lstsq(X_a, T_train, rcond=None)[0]

In [ ]:
rows = []
for name, prediction in predictions.items():
    rows.append({"Model": name,
                 "Train RMSE (µs)": 1000 * rmse(T_train, prediction),
                 "Holdout RMSE (µs)": 1000 * rmse(T_test, prediction),
                 "Holdout MAPE (%)": 100 * np.mean(np.abs(prediction - T_test) / T_test)})
comparison = pd.DataFrame(rows).set_index("Model")
comparison.round(3)

The fixed 240 KiB model gives **2.323 µs** holdout RMSE and **3.80%** MAPE, compared with **2.456 µs / 4.27%** at 256 KiB and **2.265 µs / 3.59%** at 230 KiB. Fitting the constrained boundary gives **2.218 µs / 3.33%**. The free hinge gives **2.094 µs / 2.92%**, while the affine fit gives **3.816 µs / 10.00%**.

Replacing the 256 KiB anchor with 240 KiB reduces holdout RMSE by **5.39%**. This is a reduction in prediction error, **not an SSD speedup**. The free hinge has the lowest error here, but also a less restrictive tail.

In [ ]:
main_models = ["Two-line (240 KiB)", free_line, free_hinge]
plt.figure(figsize=(9, 3.5))
for name, style in zip(main_models + ["Affine"], ["-", "--", "-.", ":"]):
    plt.plot(x, 1000 * predictions[name], style, label=name)
plt.plot(x, 1000 * T_test, ".", ms=2, alpha=.6, label="Holdout profile")
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Throughput-derived latency (µs/chunk)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 3.** Training-fitted curves against the held-out blocks. The structured models are close at this scale; their residuals expose the remaining differences.

## The approximation does not remove the residual structure.

In [ ]:
plt.figure(figsize=(9, 3.5))
for name, style in zip(main_models, ["-", "--", "-."]):
    plt.plot(x, 1000 * (predictions[name] - T_test), style, label=name)
plt.axhline(0, ls=":", lw=.7)
plt.xlabel("Logical chunk length (KiB)")
plt.ylabel("Prediction − holdout (µs/chunk)")
plt.legend(fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

**Figure 4.** Negative residuals indicate underestimated latency. The departure around 480 KiB remains under both free-boundary models. Moving the boundary improves aggregate error without explaining away the local structure.

The four lengths answer different questions:

$$\boxed{s_{99}=256,\qquad x_\rho=240,\qquad h_\star=230,\qquad s_\star=211\quad\text{KiB}.}$$

For selection algorithms, the empirical cost-per-byte minimum remains an interpretable reference. The free-boundary $T_s$ answers how closely a restricted family reproduces the table. Neither fit alone proves that a chunking policy improves the lookup-table objective; guarantees derived for $T_s$ remain conditional on that model.

### Scope

These interpretations refer to the saved native six-worker run: seven randomized blocks, 384 sizes from 1–256 KiB in 1 KiB steps and 260–768 KiB in 4 KiB steps, with three warmups and ten timed reads at each chunk-count point. The estimates are already in the CSV; this notebook only analyzes them.

The fixed candidates were nominated after inspection of this same run, and 240 KiB is the **all-seven-block** efficiency minimum. The comparison is therefore retrospective within one run, not a fresh confirmatory holdout experiment. The two free-boundary searches themselves use training data only. The optimum depends on the model, loss, grid, and reader; extrapolation beyond 768 KiB is not established.